## tinygrad: Tensor

- 无法直接`from tinygrad import Device`
- 由于tinygrad仓库是在本地，要先
  1. 显式安装`# 2. 用 editable 模式安装（必须在 tinygrad 文件夹根目录执行，注意最后的点 . ）
python -m pip install -e .`
  2. 或者按照下述方式，临时添加
- 先确认内核正常？

In [1]:
import sys
import os

tinygrad_root = r'F:\pycharm\projects\tinygrad'   # ← 确认这是 tinygrad 仓库的根目录

print("当前 sys.path 前几项：")
for p in sys.path[:10]:
    print(p)

print("\n正在添加的路径：", tinygrad_root)
print("该路径是否存在？", os.path.exists(tinygrad_root))
print("里面是否有 tinygrad 文件夹？", os.path.exists(os.path.join(tinygrad_root, "tinygrad")))

# 添加路径（推荐用 insert(0, ...) 并用绝对路径）
sys.path.insert(0, tinygrad_root)

# 再打印一次确认添加成功
print("\n添加后 sys.path[0] =", sys.path[0])

# 尝试导入
try:
    from tinygrad import Device
    print("✅ 导入成功！")
    print("默认设备:", Device.DEFAULT)
except Exception as e:
    print("❌ 还是失败，错误：", type(e).__name__, str(e))

当前 sys.path 前几项：
F:\anaconda\envs\tinygrad\python313.zip
F:\anaconda\envs\tinygrad\DLLs
F:\anaconda\envs\tinygrad\Lib
F:\anaconda\envs\tinygrad

F:\anaconda\envs\tinygrad\Lib\site-packages
__editable__.tinygrad-0.12.0.finder.__path_hook__
F:\anaconda\envs\tinygrad\Lib\site-packages\win32
F:\anaconda\envs\tinygrad\Lib\site-packages\win32\lib
F:\anaconda\envs\tinygrad\Lib\site-packages\Pythonwin

正在添加的路径： F:\pycharm\projects\tinygrad
该路径是否存在？ True
里面是否有 tinygrad 文件夹？ True

添加后 sys.path[0] = F:\pycharm\projects\tinygrad
✅ 导入成功！
默认设备: CL


In [17]:
# 首先检查 CUDA 是否安装
import subprocess
import sys
import os

def check_cuda_installation():
    """检查 CUDA 安装情况"""
    print("=" * 50)
    print("CUDA 安装检查")
    print("=" * 50)
    
    # 1. 检查 nvidia-smi
    print("\n1. 检查 NVIDIA 驱动:")
    try:
        result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, shell=True)
        if result.returncode == 0:
            print("✅ NVIDIA 驱动已安装")
            # 显示前几行
            lines = result.stdout.split('\n')
            for i, line in enumerate(lines[:8]):
                if i < 3 or "CUDA Version" in line or "Driver Version" in line:
                    print(f"   {line}")
        else:
            print("❌ NVIDIA 驱动未安装或有问题")
    except:
        print("❌ nvidia-smi 命令失败")
    
    # 2. 检查 nvcc
    print("\n2. 检查 CUDA 编译器 (nvcc):")
    try:
        result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True, shell=True)
        if result.returncode == 0:
            print("✅ CUDA Toolkit 已安装")
            lines = result.stdout.split('\n')
            for line in lines[:3]:
                print(f"   {line}")
        else:
            print("❌ nvcc 未找到")
    except:
        print("❌ CUDA Toolkit 未安装")
    
    # 3. 检查环境变量
    print("\n3. 检查环境变量:")
    env_vars = ['CUDA_PATH', 'CUDA_HOME', 'PATH']
    for var in env_vars:
        value = os.environ.get(var, '未设置')
        if var == 'PATH':
            # 只检查 PATH 中是否包含 CUDA
            if 'CUDA' in value.upper() or 'NVIDIA' in value.upper():
                print(f"✅ PATH 中包含 CUDA/NVIDIA 路径")
            else:
                print(f"⚠️  PATH 中未找到 CUDA/NVIDIA 路径")
        else:
            if value != '未设置':
                print(f"✅ {var}: {value}")
            else:
                print(f"⚠️  {var}: 未设置")

check_cuda_installation()

CUDA 安装检查

1. 检查 NVIDIA 驱动:
✅ NVIDIA 驱动已安装
   Sun Apr 12 21:15:45 2026       
   +-----------------------------------------------------------------------------------------+
   | NVIDIA-SMI 595.79                 Driver Version: 595.79         CUDA Version: 13.2     |

2. 检查 CUDA 编译器 (nvcc):
✅ CUDA Toolkit 已安装
   nvcc: NVIDIA (R) Cuda compiler driver
   Copyright (c) 2005-2026 NVIDIA Corporation
   Built on Mon_Mar__2_21:54:11_Pacific_Standard_Time_2026

3. 检查环境变量:
✅ CUDA_PATH: F:\anaconda\envs\tinygrad\Library
✅ CUDA_HOME: F:\anaconda\envs\tinygrad\Library
✅ PATH 中包含 CUDA/NVIDIA 路径


In [16]:
import os
os.environ["DEV"] = "CUDA"        # 或者 "CUDA"，目前 NV 更好

from tinygrad import Device, Tensor, nn

print("默认设备:", Device.DEFAULT)
print("可用设备:", Device._devices)

# 测试 GPU 是否正常工作
t = Tensor([1, 2, 3, 4]).to("CUDA")
print("测试张量所在设备:", t.device)
print("张量内容:", t.numpy())

默认设备: CL
可用设备: ['AMD', 'CL', 'CPU', 'CUDA', 'DISK', 'DSP', 'HIP', 'METAL', 'NPY', 'NULL', 'NV', 'PYTHON', 'QCOM', 'RDMA', 'TINYFS', 'WEBGPU']
测试张量所在设备: CUDA


AttributeError: failed to load library cuda: try setting CUDA_PATH?

In [13]:
import os
os.environ['CUDA_PATH'] = r'F:\anaconda\envs\tinygrad\Library'
os.environ['CUDA_HOME'] = r'F:\anaconda\envs\tinygrad\Library'
os.environ['PATH'] = r'F:\anaconda\envs\tinygrad\Library\bin;' + os.environ.get('PATH', '')

# 额外尝试（有时需要）
os.environ['CUDA_LIB_PATH'] = r'F:\anaconda\envs\tinygrad\Library\bin'

from tinygrad import Device, Tensor
import time

print('Device.DEFAULT =', Device.DEFAULT)

try:
    a = Tensor.randn(1024, 1024, device='CUDA').realize()
    b = Tensor.randn(1024, 1024, device='CUDA').realize()
    start = time.time()
    c = (a @ b).realize()
    print('成功！耗时:', round(time.time() - start, 4), '秒')
    print('结果设备:', c.device)
except Exception as e:
    print('仍然失败:', type(e).__name__, str(e))

Device.DEFAULT = CL
仍然失败: AttributeError failed to load library cuda: try setting CUDA_PATH?


In [5]:
from tinygrad.renderer.cstyle import OpenCLRenderer
from tinygrad.uop.ops import UOp, Ops
from tinygrad import dtypes

const = UOp(Ops.CONST, dtypes.float, arg=1.0)
add = UOp(Ops.ADD, dtypes.float, src=(const, const), arg=None)

print(add)

cl_device = Device["CL"]
print(f"获取到设备: {cl_device}")
    
# 创建渲染器
renderer = OpenCLRenderer(target=cl_device)

print(renderer.render([
  const,
  add
]))

UOp(Ops.ADD, dtypes.float, arg=None, src=(
  x0:=UOp(Ops.CONST, dtypes.float, arg=1.0, src=()),
  x0,))
获取到设备: <tinygrad.runtime.ops_cl.CLDevice object at 0x000002AF89F0C440>
__kernel void test() {
  float alu0 = (1.0f+1.0f);
}


## windows下tinygrad支持缺失，转用wsl-linux